# Загрузка данных

In [25]:
import pandas as pd

In [26]:
articles = pd.read_feather("candidate_data/articles.f")
calibration = pd.read_feather("candidate_data/calibration.f")
test = pd.read_feather("candidate_data/test.f")

d:\Глеб\Pet-projects\avito_test-task\.venv\Lib\site-packages\pandas\io\feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [27]:
articles.head()

,article_id,title,body
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://w..."
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при вх..."
2,1747,Не допустить блокировки профиля,<ol><li><p><strong>Не заводите несколько аккау...
3,1774,Оставить или удалить профиль,<p>⚡ Не удаляйте профиль с подтверждёнными дан...
4,1775,Удалить профиль,"<p>⚡ Удалить профиль не получится, если у вас ..."


In [28]:
calibration.head()

,query_id,query_text,ground_truth
0,1,Как передать товар через службу авито,1909 4234
1,2,"Можете подсказать, если заказать товар Авито д...",2865 4400
2,3,Здравствуйте. Как отправить товар через Авито.,1909
3,4,как получить деньги за возрат если продавец уж...,4400 4403
4,5,"Когда мне прийдут деньги за доставку, сегодня ...",4361


In [29]:
test.head()

,query_id,query_text
0,1,"Здравствуйте! Подскажите, пожалуйста, не могу ..."
1,2,"Здравствуйте , почему так долго доставляется в..."
2,3,"Здравствуйте,подскажите как мне отправить крос..."
3,4,Здравствуйте! В каких случаях за возврат снима...
4,5,Почему у меня доставки в несколько раз дороже ...


# EDA

In [30]:
print(f"Articles: {len(articles)}")
print(f"Calibration: {len(calibration)}")
print(f"Test: {len(test)}")

Articles: 793
Calibration: 500
Test: 500


In [31]:
articles.isna().sum()

article_id    0
title         0
body          0
dtype: int64

In [32]:
articles.sample(5)

,article_id,title,body
606,4307,Лимит бесплатных размещений,"<headline name=""Что такое лимит бесплатных раз..."
653,4354,Пройти модерацию,"<headline name=""Разместить с первого раза"" id=..."
325,3464,Неверная зарплата,<p><strong>Почему отклонили:</strong> в ваканс...
215,2892,Проблемы с номером телефона,<p><strong>Телефон используется в другом профи...
283,3056,Меня заблокировали. Что делать?,"<p>💡 <a href=""https://support.avito.ru/article..."


# Предобработка текста

## Удаление эмодзи и html из статей

In [33]:
import re

EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F" 
    "\U0001F300-\U0001F5FF"  
    "\U0001F680-\U0001F6FF" 
    "\U0001F1E0-\U0001F1FF" 
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)

In [34]:
from bs4 import BeautifulSoup

def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(" ", strip=True)
    return EMOJI_PATTERN.sub("", text)

In [61]:
articles["body_clean"] = articles.body.apply(clean_html)

articles["text"] = ( 3*(articles["title"]+ " ")
    + articles["body_clean"]
)

In [62]:
import re


def normalize(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.replace("ё", "е")
    text = text.replace("\xa0", " ")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [63]:
articles["text"] = (
    articles["text"]
    .apply(normalize)
)

In [64]:
articles.head()

,article_id,title,body,body_clean,text
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://w...",Зайдите в раздел Управление профилем . Нажмите...,имя или название компании имя или название ком...
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при вх...","Проверьте, какое сообщение вы видите при входе...","понять, что профиль заблокирован понять, что п..."
2,1747,Не допустить блокировки профиля,<ol><li><p><strong>Не заводите несколько аккау...,Не заводите несколько аккаунтов для продаж в о...,не допустить блокировки профиля не допустить б...
3,1774,Оставить или удалить профиль,<p>⚡ Не удаляйте профиль с подтверждёнными дан...,Не удаляйте профиль с подтверждёнными данными...,оставить или удалить профиль оставить или удал...
4,1775,Удалить профиль,"<p>⚡ Удалить профиль не получится, если у вас ...","Удалить профиль не получится, если у вас есть...",удалить профиль удалить профиль удалить профил...


# Модели

## BM25

In [39]:
import nltk
from nltk.corpus import stopwords
import pymorphy3

nltk.download('stopwords')

morph = pymorphy3.MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [65]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    lemmatized = []
    for token in tokens:
        if token not in russian_stopwords:
            lemma = morph.parse(token)[0].normal_form
            lemmatized.append(lemma)
    return lemmatized

bm25_corpus = [
    tokenize(text)
    for text in articles.text
]


bm25 = BM25Okapi(bm25_corpus)

## multilingual-e5-small

In [66]:
def prepare_query(text):
    return "query: " + text


def prepare_document(text):
    return "passage: " + text

In [42]:
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "intfloat/multilingual-e5-large",
    device= device
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [67]:
documents = [
    prepare_document(text)
    for text in tqdm(
        articles.text.tolist(),
        desc="Preparing documents"
    )
]


doc_embeddings = model.encode(
    documents,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

Preparing documents:   0%|          | 0/793 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

## FAISS

In [68]:
import faiss
import numpy as np

embedding_dim = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(
    np.asarray(doc_embeddings).astype("float32")
)

print("Документов в FAISS:", index.ntotal)

Документов в FAISS: 793


## Похожесть заголовка

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


title_vectorizer = TfidfVectorizer(
    ngram_range=(1,2)
)


title_matrix = title_vectorizer.fit_transform(
    articles.title.lower().fillna("")
)

In [96]:
def get_title_score(query, article_ids):

    query_vector = title_vectorizer.transform(
        [query]
    )


    indexes = []

    for aid in article_ids:

        idx = articles.index[
            articles.article_id == aid
        ][0]

        indexes.append(idx)


    scores = cosine_similarity(
        query_vector,
        title_matrix[indexes]
    )[0]


    return scores

## Топ кандидатов от bm25

In [69]:
def bm25_search(query, k=100):

    tokens = tokenize(query)

    scores = bm25.get_scores(tokens)

    top_ids = np.argsort(scores)[::-1][:k]

    return [
        {
            "article_id": articles.iloc[i].article_id,
            "score": scores[i],
            "rank": rank + 1
        }
        for rank, i in enumerate(top_ids)
    ]

## Топ кандидатов от e5

In [70]:
def dense_search(query, k=100):

    query_embedding = model.encode(
        [
            prepare_query(query)
        ],
        normalize_embeddings=True
    )

    scores, ids = index.search(
        np.asarray(query_embedding).astype("float32"),
        k
    )


    result = []

    for rank, idx in enumerate(ids[0]):

        result.append(
            {
                "article_id": articles.iloc[idx].article_id,
                "score": float(scores[0][rank]),
                "rank": rank + 1
            }
        )

    return result

## Объединение двух моделей

$$
RRF = 1/ (60+rank)
$$

In [94]:
from collections import defaultdict


def rrf_merge(
    bm25_results,
    dense_results,
    k=60
):

    scores = defaultdict(float)


    for item in bm25_results:

        scores[item["article_id"]] += 2 * (
            1 / (k + item["rank"])
        )


    for item in dense_results:

        scores[item["article_id"]] += (
            1 / (k + item["rank"])
        )


    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )


    return [
        {
            "article_id": article_id,
            "rrf_score": score
        }
        for article_id, score in ranked
    ]

## rerank

In [104]:
def minmax(values):

    values = np.array(values)

    if values.max() == values.min():
        return np.zeros_like(values)

    return (
        values - values.min()
    ) / (
        values.max() - values.min()
    )

In [108]:
def rerank(query, candidates):

    article_ids = [
        x["article_id"]
        for x in candidates
    ]


    title_scores = get_title_score(
        query,
        article_ids
    )


    rrf_scores = [
        x["rrf_score"]
        for x in candidates
    ]


    rrf_scores = minmax(
        rrf_scores
    )


    title_scores = minmax(
        title_scores
    )


    for i, item in enumerate(candidates):

        item["final_score"] = (
            rrf_scores[i]
            +
            0.05 * title_scores[i]
        )


    return sorted(
        candidates,
        key=lambda x: x["final_score"],
        reverse=True
    )

## Retrieve

In [109]:
def retrieve(query, top_k=10):

    query = normalize(query)


    bm25_candidates = bm25_search(
        query,
        k=200
    )


    dense_candidates = dense_search(
        query,
        k=200
    )


    candidates = rrf_merge(
        bm25_candidates,
        dense_candidates
    )


    candidates = rerank(
        query,
        candidates[:100]
    )


    results = [
        int(x["article_id"])
        for x in candidates[:top_k]
    ]


    return results

In [110]:
retrieve(
    "как отправить товар покупателю"
)

[1909, 1918, 4286, 4387, 2065, 3419, 4403, 4396, 4234, 4400]

In [111]:
articles[articles['article_id'] == retrieve(
    "как отправить товар покупателю"
)[0]]

,article_id,title,body,body_clean,text
48,1909,Отправить заказ,"<ol><li><p>У вас будет <strong>2 рабочих дня, ...","У вас будет 2 рабочих дня, чтобы отправить тов...",отправить заказ отправить заказ отправить зака...


# Оценка решения

In [112]:
def average_precision_at_k(
    predicted,
    actual,
    k=10
):

    actual = set(actual)

    predicted = predicted[:k]


    score = 0
    hits = 0


    for i, article_id in enumerate(
        predicted,
        start=1
    ):

        if article_id in actual:

            hits += 1

            score += hits / i


    if len(actual) == 0:
        return 0


    return score / min(
        len(actual),
        k
    )

def evaluate_map10(df):

    scores = []


    for _, row in df.iterrows():

        prediction = retrieve(
            row.query_text,
            top_k=10
        )


        target = list(
            map(
                int,
                row.ground_truth.split()
            )
        )


        score = average_precision_at_k(
            prediction,
            target,
            k=10
        )


        scores.append(score)


    return np.mean(scores)

In [113]:
def recall_at_k(predicted, actual, k=10):

    predicted = set(predicted[:k])
    actual = set(actual)

    return len(predicted & actual) / len(actual)

def evaluate_recall10(df):

    scores = []

    for _, row in df.iterrows():

        prediction = retrieve(
            row.query_text,
            top_k=10
        )

        target = list(
            map(
                int,
                row.ground_truth.split()
            )
        )

        scores.append(
            recall_at_k(
                prediction,
                target,
                10
            )
        )

    return np.mean(scores)

In [114]:
map10 = evaluate_map10(calibration)
print(f"MAP@10: {map10:.4f}")
recall10 = evaluate_recall10(calibration)
print(f"Recall@10: {recall10:.4f}")

MAP@10: 0.3666
Recall@10: 0.6955


MAP@10: 0.3666

Recall@10: 0.6955

# Предсказания на тесте

In [136]:
test_predictions = []


for query in test.query_text:

    result = retrieve(
        query,
        top_k=10
    )

    test_predictions.append(
        " ".join(
            map(str, result)
        )
    )

In [137]:
answer = pd.DataFrame(
    {
        "query_id": test.query_id,
        "answer": test_predictions
    }
)

In [138]:
answer.head()

,query_id,answer
0,1,3565 1960 2196 4331 4326 4261 2962 2943 2661 1899
1,2,4009 1923 3149 4403 4407 2944 2802 2408 3209 2521
2,3,2831 1909 4234 4400 4308 4361 4431 4219 4396 1960
3,4,4400 2865 4219 3006 2866 2831 1966 2646 4331 4234
4,5,4388 2698 4214 4320 2232 4045 4219 4391 4265 4409


In [140]:
answer.to_csv(
    "answer.csv",
    index=False
)